# Exploratory Infrastructure Analysis — Ottawa EV Charging

## Purpose

This notebook explores three infrastructure datasets:

- `stations_detail.csv`
- `ports_detail.csv`
- `plugs_detail.csv`

The goal is to determine whether these datasets provide reliable station-level context that can later enrich the cleaned Ottawa EV charging-session analysis.

**Important:** The cleaned EV charging usage dataset is not modified or merged in this notebook. This notebook is limited to infrastructure exploration, validation, and preparation.


## 1. Libraries and Display Settings

Import the libraries used for data inspection and later geospatial work.


In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import MarkerCluster
import geopandas as gpd
from pyproj import Transformer

plt.rcParams["figure.figsize"] = (20, 7)

import requests

## 2. Load the Infrastructure Datasets

Load the station, port, and plug datasets. `low_memory=False` is used because the plug dataset contains a mixed-type column (`ParkingInfo`).


In [3]:
# Load cleaned dataset
# Load datasets
plugs = pd.read_csv(
    "plugs_detail.csv",
    low_memory=False
)

ports = pd.read_csv(
    "ports_detail.csv",
    low_memory=False
)

stations = pd.read_csv(
    "stations_detail.csv",
    low_memory=False
)



In [8]:
plugs

,StationId,Code,IconUrl,Level,Status,LastTimeUsed,Network,NetworkName,NetworkIconUrl,Count,...,Amp,Kw,Volt,StatusTime,LiveStatusLastUpdate,PlugPassportEnabled,FlexFee,Name,DetailedStatusInfo_String,DetailedStatusInfo_Style
0,100020,5,https://evclstorage.blob.core.windows.net/ch-c...,2,1,Last used less than 24 hours ago,1,Blink,https://evclstorage.blob.core.windows.net/ch-n...,2,...,40 A,10.8 kW,207 V,2025-06-16T21:15:52.997Z,2025-06-16T22:11:54.000Z,True,0.75,EV Plug (J1772),2/2 Available,AtLeast1AvailableStyle
1,100061,5,https://evclstorage.blob.core.windows.net/ch-c...,2,1,Last used less than 24 hours ago,2,ChargePoint,https://evclstorage.blob.core.windows.net/ch-n...,4,...,27 A,6.6 kW,240 V,2025-06-16T21:50:00.110Z,2025-06-16T22:38:36.000Z,True,0.75,EV Plug (J1772),2/4 Available,AtLeast1AvailableStyle
2,100070,5,https://evclstorage.blob.core.windows.net/ch-c...,2,1,Last used 2 days ago,2,ChargePoint,https://evclstorage.blob.core.windows.net/ch-n...,2,...,27 A,6.6 kW,240 V,2025-06-11T11:16:37.473Z,2025-06-16T23:27:58.000Z,True,0.75,EV Plug (J1772),2/2 Available,AtLeast1AvailableStyle
3,100077,7,https://evclstorage.blob.core.windows.net/ch-c...,2,0,NaN,12,Tesla,https://evclstorage.blob.core.windows.net/ch-n...,2,...,NaN,NaN,NaN,2023-06-26T17:56:45.510Z,NaN,False,0.00,NACS (Tesla),2 Ports,RegularStyle
4,100084,8,https://evclstorage.blob.core.windows.net/ch-c...,3,5,NaN,17,NaN,NaN,1,...,NaN,NaN,NaN,NaN,NaN,False,0.00,CCS & CHAdeMO,1 Coming Soon,ConstructionStyle
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33260,99991,5,https://evclstorage.blob.core.windows.net/ch-c...,2,0,Last used less than 24 hours ago,2,ChargePoint,https://evclstorage.blob.core.windows.net/ch-n...,1,...,27 A,6.6 kW,240 V,2025-06-16T22:05:37.083Z,2025-06-16T22:45:37.000Z,True,0.75,EV Plug (J1772),1 Port,RegularStyle
33261,99991,5,https://evclstorage.blob.core.windows.net/ch-c...,2,1,Last used less than 24 hours ago,2,ChargePoint,https://evclstorage.blob.core.windows.net/ch-n...,4,...,27 A,6.6 kW,240 V,2025-06-16T21:04:27.317Z,2025-06-16T22:45:37.000Z,True,0.75,EV Plug (J1772),3/4 Available,AtLeast1AvailableStyle
33262,99993,5,https://evclstorage.blob.core.windows.net/ch-c...,2,1,Last used less than 24 hours ago,2,ChargePoint,https://evclstorage.blob.core.windows.net/ch-n...,4,...,27 A,6.6 kW,240 V,2025-06-16T19:32:12.227Z,2025-06-16T22:53:39.000Z,True,0.00,EV Plug (J1772),4/4 Available,AtLeast1AvailableStyle
33263,99993,5,https://evclstorage.blob.core.windows.net/ch-c...,2,4,Last used over 2 weeks ago,2,ChargePoint,https://evclstorage.blob.core.windows.net/ch-n...,2,...,27 A,6.6 kW,240 V,2024-10-08T15:07:05.020Z,2025-06-16T22:53:39.000Z,True,0.00,EV Plug (J1772),2 Offline,OfflineStyle


In [9]:
plugs.shape

(33265, 27)

### Mixed-Type Warning Check

Pandas identified column position `16` as containing mixed data types. Because Python uses zero-based indexing, `plugs.columns[16]` refers to the 17th column, which is `ParkingInfo`.

The field is almost entirely missing and is not required for the current analysis, so no further cleaning is needed at this stage.


In [7]:
# Check which column caused the mixed-type warning
print(plugs.columns[16])

ParkingInfo


In [6]:
# Check the data types in that column
plugs[plugs.columns[16]].apply(type).value_counts()

ParkingInfo
<class 'float'>    33257
<class 'str'>          8
Name: count, dtype: int64

In [10]:
ports

,StationId,Code,portId,netPortId,status,displayName,statusInfo_String,statusInfo_Style,activationType,lastTimeUsed
0,100020,5,844381,1800efe3-965f-4a62-955f-f2ce868cf58c,1,-W-2208-007456-1,Available,AvailableStyle,Complete,Last used an hour ago
1,100020,5,1041640,4e7ecded-1af7-5147-982d-640e53761058,1,-W-2220-009072-1,Available,AvailableStyle,Complete,Last used an hour ago
2,100061,5,723742,9066521,2,KELLOGG SQUARE/KS-111-35AE (Port 1),In use,InUseStyle,Partial,NaN
3,100061,5,723743,9066531,1,KELLOGG SQUARE/KS-111-35AE (Port 2),Available,AvailableStyle,Complete,Last used yesterday
4,100061,5,959112,23572691,2,KELLOGG SQUARE/KS-111-35BB (Port 1),In use,InUseStyle,Partial,NaN
...,...,...,...,...,...,...,...,...,...,...
97603,99993,5,945561,1022591,1,ALECTRA/DERRY EMP2 (Port 2),Available,AvailableStyle,Complete,Last used 11 hours ago
97604,99993,5,945562,2445411,4,ALECTRA/DERRY EMP3 (Port 1),Offline,OfflineStyle,Partial,Last used over 2 weeks ago
97605,99993,5,945563,2445421,4,ALECTRA/DERRY EMP3 (Port 2),Offline,OfflineStyle,Partial,Last used over 2 weeks ago
97606,99999,5,614310,NaN,0,Port 1,NaN,NaN,NaN,NaN


In [11]:
stations

,Id,IdAsString,IsShared,LocName,Address,StreetNo,Street,City,ProvCode,ProvName,...,PriceNumber,PriceString,PriceStringInfo,PriceStringSum,LocDesc,AccessType,StatusTime,PaymentMethods,IsPassportEnabled,LocScore
0,100020,100020,False,The Clara,"225 South Linder Road, Eagle, ID, 83616",225.0,South Linder Road,Eagle,ID,Idaho,...,NaN,$0.49/session; $0.49/kWh,NaN,NaN,NaN,Public,2025-06-16T22:03:31.403Z,"ChargeHub App, Network App, Network RFID",True,NaN
1,100061,100061,False,Kellogg Square,"111 East Kellogg Boulevard, Saint Paul, MN, 55101",111.0,East Kellogg Boulevard,Saint Paul,MN,Minnesota,...,NaN,Time\n 2 USD per hour\nParking \n 0 USD pe...,NaN,NaN,In parking garage,Public,2025-06-16T22:07:24.620Z,"ChargeHub App, Network App, Network RFID, Visa...",True,NaN
2,100070,100070,False,Chazy Public Library,"1329 Fiske Road, Chazy, NY, 12921",1329.0,Fiske Road,Chazy,NY,New York,...,NaN,Parking \n 0 USD per hour for first 20 mins ...,NaN,NaN,Rear parking lot of chazy library,Public,2025-06-15T02:05:12.000Z,"ChargeHub App, Network App, Network RFID, Visa...",True,NaN
3,100077,100077,False,Stoller Vineyards Experience Center - Tesla De...,"16161 Northeast McDougall Road, Dayton, OR, 97114",16161.0,Northeast McDougall Road,Dayton,OR,Oregon,...,NaN,Cost: Free,NaN,NaN,NaN,Restricted,2023-06-26T17:56:45.510Z,No network activation required,False,NaN
4,100084,100084,False,Tatchun Centre General Store,"35607 Klondike Highway, Carmacks, YT, Y0B 1C0",35607.0,Klondike Highway,Carmacks,YT,Yukon,...,NaN,Cost: Unknown,NaN,NaN,NaN,Public,NaN,Unknown activation method,False,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27229,99986,99986,False,Best Western Inn At The Meadows,"1215 North Hayden Meadows Drive, Portland, OR,...",1215.0,North Hayden Meadows Drive,Portland,OR,Oregon,...,NaN,Energy\n 0.22 USD per kWh\nParking \n 0.5 ...,NaN,NaN,Best Western Inn at the Meadows Parking Lot,Public,2023-12-07T14:15:36.607Z,"ChargeHub App, Network App, Network RFID, Visa...",True,NaN
27230,99990,99990,False,"Canadian Tire - London, ON","1125 Wellington Road, London, ON, N6E 1M1",1125.0,Wellington Road,London,ON,Ontario,...,NaN,"Guest and Pass Members: 1-90Kw $0.27/min, 1-35...",NaN,NaN,NaN,Public,NaN,"Network App, Network RFID, Visa, Mastercard",False,2.8
27231,99991,99991,False,Auburn Hills Community Center,"1827 North Squirrel Road, Auburn Hills, MI, 48326",1827.0,North Squirrel Road,Auburn Hills,MI,Michigan,...,NaN,Parking \n 1 USD per hour for first 180 mins...,NaN,NaN,Near tennis court,Public,2025-06-16T22:05:37.083Z,"ChargeHub App, Network App, Network RFID, Visa...",True,NaN
27232,99993,99993,False,Alectra Corporate Head Office,"2185 Derry Road West, Mississauga, ON, L5N",2185.0,Derry Road West,Mississauga,ON,Ontario,...,0.0,Cost: Free,NaN,NaN,,Public,2025-06-16T21:17:02.810Z,"ChargeHub App, Network App, Network RFID, Visa...",True,NaN


## 3. Geographic Scope

The source data includes charging stations from both Canada and the United States. The project scope is Canada, so the station dataset is first filtered to Canada and then narrowed to Ottawa, Ontario.


In [13]:
# Check countries represented in the station dataset
stations["CountryName"].value_counts()

CountryName
United States    14265
Canada           12967
Name: count, dtype: int64

In [14]:
# Check countries represented in the station dataset
stations["ProvName"].value_counts()

ProvName
Québec                       5211
Ontario                      3599
New York                     2541
British Columbia             2121
Massachusetts                2062
Washington                   1767
Michigan                     1485
Oregon                       1195
Illinois                      900
Minnesota                     894
Alberta                       709
Wisconsin                     698
Maine                         451
Vermont                       409
Connecticut                   384
Nova Scotia                   287
New Brunswick                 260
Manitoba                      259
New Hampshire                 251
Iowa                          218
Idaho                         208
Prince Edward Island          174
Saskatchewan                  168
Rhode Island                  157
Montana                       149
Newfoundland and Labrador     132
South Dakota                  116
North Dakota                   86
Wyoming                        76
Penns

In [15]:
# Create a Canada-only station dataset
stations_canada = stations[
    stations["CountryName"] == "Canada"
].copy()

# Check station counts by province
stations_canada["ProvName"].value_counts()

ProvName
Québec                       5211
Ontario                      3599
British Columbia             2121
Alberta                       709
Nova Scotia                   287
New Brunswick                 260
Manitoba                      259
Prince Edward Island          174
Saskatchewan                  168
Newfoundland and Labrador     132
Yukon                          37
Northwest Territory            10
Name: count, dtype: int64

## 4. Ottawa Station Subset

Filter the Canadian station dataset to Ottawa, Ontario.

The Ottawa subset contains **218 charging locations** with station-level details such as location name, address, coordinates, port counts, network, access, and pricing.


In [16]:
# Filter to Ontario and Ottawa
stations_ottawa = stations_canada[
    (stations_canada["ProvName"] == "Ontario") &
    (stations_canada["City"] == "Ottawa")
].copy()

# Check the result
stations_ottawa.shape

(218, 34)

In [18]:
stations_ottawa.info()

<class 'pandas.core.frame.DataFrame'>
Index: 218 entries, 112 to 27163
Data columns (total 34 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Id                 218 non-null    int64  
 1   IdAsString         218 non-null    int64  
 2   IsShared           218 non-null    bool   
 3   LocName            218 non-null    object 
 4   Address            218 non-null    object 
 5   StreetNo           212 non-null    float64
 6   Street             218 non-null    object 
 7   City               218 non-null    object 
 8   ProvCode           218 non-null    object 
 9   ProvName           218 non-null    object 
 10  CountryCode        218 non-null    object 
 11  CountryName        218 non-null    object 
 12  Zip                218 non-null    object 
 13  Lat                218 non-null    float64
 14  Long               218 non-null    float64
 15  Phone              168 non-null    object 
 16  Web                165 non-

## 5. Initial Ottawa Data Quality Checks

Check station IDs, duplicate addresses, port-count distribution, charging networks, access types, and pricing information.

Duplicate addresses are not automatically treated as duplicate stations because multiple charging records can legitimately exist at the same address.


In [19]:
# Check duplicate station IDs
print("Duplicate Id values:", stations_ottawa["Id"].duplicated().sum())

# Check duplicate addresses
print("Duplicate addresses:", stations_ottawa["Address"].duplicated().sum())

# Review port counts
print(stations_ottawa["TotalPorts"].describe())

# Review charging networks
print(stations_ottawa["NetworkName"].value_counts(dropna=False).head(10))

# Review access types
print(stations_ottawa["AccessType"].value_counts(dropna=False))

# Review pricing information
print(stations_ottawa["PriceString"].value_counts(dropna=False).head(10))

Duplicate Id values: 0
Duplicate addresses: 8
count    218.000000
mean       3.798165
std        4.514758
min        1.000000
25%        2.000000
50%        2.000000
75%        4.000000
max       50.000000
Name: TotalPorts, dtype: float64
NetworkName
Flo                   68
ChargePoint           46
NaN                   43
SWTCH                 10
Unknown                6
ChargeLab              6
EVStart                6
AmbuEnergy             6
Tesla                  5
Circuit Électrique     5
Name: count, dtype: int64
AccessType
Public        203
Restricted     15
Name: count, dtype: int64
PriceString
$2.00 / hr                                                                                       49
Cost: Unknown                                                                                    42
Cost: Free                                                                                       26
$20.00 / hr                                                                             

In [20]:
# Inspect the duplicate addresses
duplicate_addresses = stations_ottawa[
    stations_ottawa["Address"].duplicated(keep=False)
].sort_values("Address")

duplicate_addresses[
    [
        "Id",
        "LocName",
        "Address",
        "TotalPorts",
        "NetworkName",
        "PriceString"
    ]
]

,Id,LocName,Address,TotalPorts,NetworkName,PriceString
6795,130558,Myers Orleans Nissan,"1452 Youville Drive, Ottawa, ON, K1C 2X8",1,Flo,$20.00 / hr
9651,143966,Myers Orleans Nissan,"1452 Youville Drive, Ottawa, ON, K1C 2X8",2,Flo,$5.00 / hr
5363,123942,Myers Orleans GMC - Indoor,"1875 St Joseph Boulevard, Ottawa, ON, K1C 7J2",1,Flo,$5.00 / hr
9132,141768,Myers Orleans GMC,"1875 St Joseph Boulevard, Ottawa, ON, K1C 7J2",1,Flo,$20.00 / hr
9647,143958,Myers Orleans GMC - Indoor,"1875 St Joseph Boulevard, Ottawa, ON, K1C 7J2",1,Flo,Cost: Free
10800,149509,Bantree Warehouses,"2101 Bantree Street, Ottawa, ON, K1B 4X3",1,Flo,$3.00 / hr
10801,149510,Gentek Building Products,"2101 Bantree Street, Ottawa, ON, K1B 4X3",1,Flo,$3.00 / hr
10798,149507,Bantree Warehouses - 2105 Bantree Street Ottaw...,"2105 Bantree Street, Ottawa, ON, K1B 4X3",1,Flo,$3.00 / hr
10799,149508,Gentek Building Products,"2105 Bantree Street, Ottawa, ON, K1B 4X3",1,Flo,$3.00 / hr
5111,122712,Giant Tiger Head Office Ottawa Underground,"2480 Walkley Road, Ottawa, ON, K1G 6A9",8,Flo,$2.00 / hr


In [21]:
# Inspect locations with unusually high port counts
stations_ottawa[
    stations_ottawa["TotalPorts"] >= 10
][
    [
        "LocName",
        "Address",
        "TotalPorts",
        "AvailablePorts",
        "NetworkName"
    ]
].sort_values(
    "TotalPorts",
    ascending=False
)

,LocName,Address,TotalPorts,AvailablePorts,NetworkName
10576,Chamber of Marine Commerce,"300 Sparks Street, Ottawa, ON, K1A 0J6",50,27,SWTCH
14856,Carleton University,"1125 Colonel By Drive, Ottawa, ON, K1S 5B6",20,0,NaN
15779,9551 Ring Road,"9551 Ring Road, Ottawa, ON, K1H 8L6",20,17,SWTCH
9494,Preston Square,"343 Preston, Ottawa, ON, K1S 1N4",18,17,Flo
15823,"Ottawa - St. Laurent Blvd, ON Supercharger","1055 Saint Laurent Boulevard, Ottawa, ON, K1K 3B1",16,0,Tesla
9910,The Royale - Commercial,"245 Rideau Street, Ottawa, ON, K1N 5Y2",15,15,Flo
7323,The Business Inn & Suites (Guests Only),"180 MacLaren Street, Ottawa, ON, K2P0L3",12,0,AmbuEnergy
11273,Rosefellow Building A,"405 Huntmar Drive, Ottawa, ON, K2T 0H7",12,0,Hypercharge
14068,"Ottawa - Campeau Dr, ON Supercharger","8605 Campeau Drive, Ottawa, ON, K2T 0K5",12,0,Tesla
16033,"Ottawa - Boundary Rd, ON Supercharger","5329 Boundary Road, Ottawa, ON, K4B 1P6",12,0,Tesla


In [22]:
# Look for City of Ottawa charging locations
city_of_ottawa_stations = stations_ottawa[
    stations_ottawa["LocName"].str.contains(
        "Ottawa",
        case=False,
        na=False
    )
].copy()

city_of_ottawa_stations[
    [
        "Id",
        "LocName",
        "Address",
        "TotalPorts",
        "AvailablePorts",
        "NetworkName",
        "PriceString"
    ]
]

,Id,LocName,Address,TotalPorts,AvailablePorts,NetworkName,PriceString
956,104061,Ottawa - Blackburn Arena,"200 Glen Park Drive, Ottawa, ON, K1B 5A3",2,2,Flo,$2.00 / hr
1363,108549,Ikea Ottawa ON,"2685 Iris Street, Ottawa, ON, K2C 3S4",8,5,ChargePoint,Parking \n 0 CAD per hour for first 120 mins...
2396,112180,HTS Ottawa,"830 Campbell Avenue, Ottawa, ON, K2A 2C2",2,2,ChargePoint,$2.00 / hr
2759,113173,uOttawa,"55 Laurier Avenue East, Ottawa, ON, K1N 6N5",4,0,NaN,Cost: Unknown
2760,113174,uOttawa,"200 Lees Avenue, Ottawa, ON, K1N 6N5",6,0,NaN,Cost: Unknown
...,...,...,...,...,...,...,...
24257,81671,Ottawa Glebe Parking Garage,"170 Second Avenue, Ottawa, ON, K1S 2H6",2,2,Flo,$2.00 / hr
24341,82010,VIA Rail Ottawa EVC,"200 Tremblay Road, Ottawa, ON, K1G 3H5",2,0,Circuit Électrique,$2.00 / hr
26024,90692,"Farm Boy - Train Yards - Ottawa, ON","830 Belfast Road, Ottawa, ON, K1G 0Z4",2,1,Flo,$2.00 / hr
26089,90992,Canadian Tire Ottawa East,"330 Coventry Road, Ottawa, ON, K1K 4S3",4,0,Electrify Canada,"Guest and Pass Members: 1-90Kw $0.27/min, 1-35..."


## 6. Address Matching Examples

Two addresses from the cleaned Ottawa usage dataset are checked manually against the infrastructure dataset.

These examples demonstrate that infrastructure records can provide additional real-world context. For example, **190 Richmond Road** is identified as a **Real Canadian Superstore** location.

This type of context may later help explain differences in charging-session duration and usage behaviour.


In [23]:
# Look for 190 Richmond in the Ottawa station dataset
stations_ottawa[
    stations_ottawa["Address"].str.contains(
        "190 Richmond",
        case=False,
        na=False
    )
][
    [
        "Id",
        "LocName",
        "Address",
        "TotalPorts",
        "AvailablePorts",
        "NetworkName",
        "PriceString"
    ]
]

,Id,LocName,Address,TotalPorts,AvailablePorts,NetworkName,PriceString
2846,113451,Real Canadian Superstore,"190 Richmond Road, Ottawa, ON, K1Z 6W6",2,1,Flo,$2.00 / hr


In [24]:
# Look for 245 Crichton in the Ottawa station dataset
stations_ottawa[
    stations_ottawa["Address"].str.contains(
        "245 Crichton",
        case=False,
        na=False
    )
][
    [
        "Id",
        "LocName",
        "Address",
        "TotalPorts",
        "AvailablePorts",
        "NetworkName",
        "PriceString"
    ]
]

,Id,LocName,Address,TotalPorts,AvailablePorts,NetworkName,PriceString
2602,112900,293 Crichton Street,"245 Crichton Street, Ottawa, ON, K1M 1W3",2,1,Flo,$2.00 / hr


## 7. Link Ottawa Stations to Port and Plug Records

`stations_ottawa["Id"]` is the station key.

Both the port and plug datasets use `StationId`, allowing the Ottawa station IDs to filter the related records.

All **218 Ottawa stations** are represented in both supporting datasets.


In [25]:
# Get the list of Ottawa station IDs
ottawa_station_ids = stations_ottawa["Id"]

# Filter ports to Ottawa stations only
ports_ottawa = ports[
    ports["StationId"].isin(ottawa_station_ids)
].copy()

# Filter plugs to Ottawa stations only
plugs_ottawa = plugs[
    plugs["StationId"].isin(ottawa_station_ids)
].copy()

In [26]:
# Check the shapes of the Ottawa subsets
print("Ottawa stations:", stations_ottawa.shape)
print("Ottawa ports:", ports_ottawa.shape)
print("Ottawa plugs:", plugs_ottawa.shape)

Ottawa stations: (218, 34)
Ottawa ports: (832, 10)
Ottawa plugs: (252, 27)


In [27]:
# Check the number of unique station IDs represented
print(
    "Unique StationIds in ports:",
    ports_ottawa["StationId"].nunique()
)

print(
    "Unique StationIds in plugs:",
    plugs_ottawa["StationId"].nunique()
)

Unique StationIds in ports: 218
Unique StationIds in plugs: 218


In [28]:
ports_ottawa.info()
plugs_ottawa.info()

<class 'pandas.core.frame.DataFrame'>
Index: 832 entries, 383 to 97306
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   StationId          832 non-null    int64 
 1   Code               832 non-null    int64 
 2   portId             832 non-null    int64 
 3   netPortId          572 non-null    object
 4   status             832 non-null    int64 
 5   displayName        832 non-null    object
 6   statusInfo_String  620 non-null    object
 7   statusInfo_Style   620 non-null    object
 8   activationType     324 non-null    object
 9   lastTimeUsed       479 non-null    object
dtypes: int64(4), object(6)
memory usage: 71.5+ KB
<class 'pandas.core.frame.DataFrame'>
Index: 252 entries, 135 to 33170
Data columns (total 27 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   StationId                  252 non-null    int64  
 1   Code

In [29]:
ports_ottawa.columns
plugs_ottawa.columns

Index(['StationId', 'Code', 'IconUrl', 'Level', 'Status', 'LastTimeUsed',
       'Network', 'NetworkName', 'NetworkIconUrl', 'Count', 'PriceString',
       'ChargingRateUnit', 'ChargingRateInfo', 'ChargingRate', 'ParkingRate',
       'ParkingUnit', 'ParkingInfo', 'Amp', 'Kw', 'Volt', 'StatusTime',
       'LiveStatusLastUpdate', 'PlugPassportEnabled', 'FlexFee', 'Name',
       'DetailedStatusInfo_String', 'DetailedStatusInfo_Style'],
      dtype='object')

## 8. Create Station-Level Port and Plug Summaries

The port and plug tables contain multiple rows per station, so they are summarized to one row per `StationId` before being merged with the station table.

This avoids creating duplicate station rows during the merge.


In [30]:
# Summarize port information by station
ports_summary = (
    ports_ottawa.groupby("StationId")
    .agg(
        port_count=("portId", "count"),
        ports_used=("lastTimeUsed", "count")
    )
    .reset_index()
)

ports_summary.head()

,StationId,port_count,ports_used
0,32381,2,2
1,32997,1,0
2,32998,3,0
3,33390,5,5
4,34495,1,1


In [31]:
# Summarize plug information by station
plugs_summary = (
    plugs_ottawa.groupby("StationId")
    .agg(
        plug_types=("Name", "nunique"),
        max_level=("Level", "max"),
        plug_count=("Count", "sum")
    )
    .reset_index()
)

plugs_summary.head()

,StationId,plug_types,max_level,plug_count
0,32381,1,2,2
1,32997,1,2,1
2,32998,1,2,3
3,33390,1,2,5
4,34495,1,3,1


In [32]:
# Merge port information into the station dataset
stations_ottawa_full = stations_ottawa.merge(
    ports_summary,
    left_on="Id",
    right_on="StationId",
    how="left"
)

# Merge plug information
stations_ottawa_full = stations_ottawa_full.merge(
    plugs_summary,
    left_on="Id",
    right_on="StationId",
    how="left",
    suffixes=("_ports", "_plugs")
)

stations_ottawa_full.head()

,Id,IdAsString,IsShared,LocName,Address,StreetNo,Street,City,ProvCode,ProvName,...,PaymentMethods,IsPassportEnabled,LocScore,StationId_ports,port_count,ports_used,StationId_plugs,plug_types,max_level,plug_count
0,100539,100539,False,NRC Automotive and Surface Transportation,"2320 Lester Road, Ottawa, ON, K1V 1S2",2320.0,Lester Road,Ottawa,ON,Ontario,...,"ChargeHub App, Network App, Network RFID",True,NaN,100539,2,2,100539,1,2,2
1,104061,104061,False,Ottawa - Blackburn Arena,"200 Glen Park Drive, Ottawa, ON, K1B 5A3",200.0,Glen Park Drive,Ottawa,ON,Ontario,...,"Network App, Network RFID",False,2.7,104061,2,2,104061,1,2,2
2,104887,104887,False,Merivale Mall,"1642 Merivale Road, Ottawa, ON, K2G 4A1",1642.0,Merivale Road,Ottawa,ON,Ontario,...,"ChargeHub App, Network App, Network RFID",True,3.6,104887,2,2,104887,1,2,2
3,105099,105099,False,METRO - Lincoln Fields Store,"2525 Carling Avenue, Ottawa, ON, K2B 7Z2",2525.0,Carling Avenue,Ottawa,ON,Ontario,...,"Network App, Network RFID",False,2.6,105099,4,2,105099,1,2,4
4,107029,107029,False,525 Legget Drive,"525 Legget Drive, Ottawa, ON, K2K 2W2",525.0,Legget Drive,Ottawa,ON,Ontario,...,"ChargeHub App, Network App, Network RFID, Visa...",True,NaN,107029,2,1,107029,1,2,2


In [33]:
stations_ottawa_full[
    [
        "LocName",
        "Address",
        "TotalPorts",
        "port_count",
        "plug_count",
        "plug_types",
        "max_level"
    ]
].head(20)

,LocName,Address,TotalPorts,port_count,plug_count,plug_types,max_level
0,NRC Automotive and Surface Transportation,"2320 Lester Road, Ottawa, ON, K1V 1S2",2,2,2,1,2
1,Ottawa - Blackburn Arena,"200 Glen Park Drive, Ottawa, ON, K1B 5A3",2,2,2,1,2
2,Merivale Mall,"1642 Merivale Road, Ottawa, ON, K2G 4A1",2,2,2,1,2
3,METRO - Lincoln Fields Store,"2525 Carling Avenue, Ottawa, ON, K2B 7Z2",4,4,4,1,2
4,525 Legget Drive,"525 Legget Drive, Ottawa, ON, K2K 2W2",2,2,2,1,2
5,Ikea Ottawa ON,"2685 Iris Street, Ottawa, ON, K2C 3S4",8,8,8,1,2
6,Kids & Company,"110 Michael Cowpland Drive, Ottawa, ON, K2M 1X3",1,1,1,1,2
7,Qualicum Center,"2936 Baseline Road, Ottawa, ON, K2H 1B3",2,2,2,1,2
8,337 Via Chianti Grove,"337 Via Chianti Grove, Ottawa, ON, K2J 6M4",2,2,2,1,2
9,605 Longfields Drive Station 1,"605 Longfields Drive, Ottawa, ON, K2J 6B7",2,2,2,1,2


## 9. Validate Station Capacity Across the Three Datasets

Compare:

- `TotalPorts` from the station table
- `port_count` derived from the port table
- `plug_count` derived from the plug table

The three sources agree for **217 of 218 Ottawa stations**, providing strong confidence in the infrastructure capacity data.

The only mismatch is **Indigo - Pinecrest**, where the station table reports 2 ports while both the port and plug tables report 6.


In [34]:
# Check whether the station, port and plug counts agree

stations_ottawa_full["ports_match"] = (
    stations_ottawa_full["TotalPorts"] ==
    stations_ottawa_full["port_count"]
)

stations_ottawa_full["plugs_match"] = (
    stations_ottawa_full["TotalPorts"] ==
    stations_ottawa_full["plug_count"]
)

# Count how many stations match
print(
    "TotalPorts matches port_count:",
    stations_ottawa_full["ports_match"].sum(),
    "of",
    len(stations_ottawa_full)
)

print(
    "TotalPorts matches plug_count:",
    stations_ottawa_full["plugs_match"].sum(),
    "of",
    len(stations_ottawa_full)
)

TotalPorts matches port_count: 217 of 218
TotalPorts matches plug_count: 217 of 218


In [35]:
# Display stations where the capacity counts do not agree
capacity_mismatches = stations_ottawa_full[
    (~stations_ottawa_full["ports_match"]) |
    (~stations_ottawa_full["plugs_match"])
]

capacity_mismatches[
    [
        "LocName",
        "Address",
        "TotalPorts",
        "port_count",
        "plug_count",
        "plug_types",
        "max_level"
    ]
]

,LocName,Address,TotalPorts,port_count,plug_count,plug_types,max_level
139,Indigo - Pinecrest,"2735 Iris Street, Ottawa, ON, K2C 3Z7",2,6,6,1,2


## 10. Charging Level, Plug Type, and Technical Specifications

Inspect charger level, connector type, power, voltage, and amperage.

Ottawa's infrastructure is dominated by **Level 2** charging and **J1772** connectors, with a smaller group of higher-power DC fast chargers.


In [36]:
# Check charging levels
print("Charging Levels:")
print(plugs_ottawa["Level"].value_counts().sort_index())

print("\nPlug Types:")
print(plugs_ottawa["Name"].value_counts())

print("\nPower (kW) values:")
print(plugs_ottawa["Kw"].value_counts(dropna=False).head(20))

print("\nVoltage values:")
print(plugs_ottawa["Volt"].value_counts(dropna=False).head(20))

print("\nAmp values:")
print(plugs_ottawa["Amp"].value_counts(dropna=False).head(20))

Charging Levels:
Level
1      2
2    216
3     34
Name: count, dtype: int64

Plug Types:
Name
EV Plug (J1772)    210
CCS & CHAdeMO       19
NACS (Tesla)        10
CCS                  5
CCS & CCS            3
CHAdeMO              2
NEMA 5-20            2
CCS & Tesla          1
Name: count, dtype: int64

Power (kW) values:
Kw
NaN        61
6.6 kW     50
6.2 kW     37
7.2 kW     28
50 kW      19
16.6 kW    11
9.6 kW     11
6.7 kW     10
10 kW       3
8.32 kW     3
325 kW      2
200 kW      2
12 kW       2
7.7 kW      2
7.68 kW     2
150 kW      2
6 kW        1
120 kW      1
11.5 kW     1
5 kW        1
Name: count, dtype: int64

Voltage values:
Volt
NaN       86
240 V     80
208 V     62
400 V     21
1000 V     2
225 V      1
Name: count, dtype: int64

Amp values:
Amp
NaN      83
30 A     58
27 A     45
125 A    15
80 A     12
32 A     11
40 A      8
29 A      4
31 A      3
48 A      3
200 A     2
375 A     2
812 A     2
20 A      1
625 A     1
180 A     1
50 A      1
Name: count, dtype: 

In [37]:
# Display the main plug-level technical fields
plugs_ottawa[
    [
        "StationId",
        "Name",
        "Level",
        "Count",
        "Kw",
        "Volt",
        "Amp",
        "NetworkName",
        "PriceString"
    ]
].head(25)

,StationId,Name,Level,Count,Kw,Volt,Amp,NetworkName,PriceString
135,100539,EV Plug (J1772),2,2,12 kW,240 V,50 A,ChargePoint,Cost: Free
1178,104061,EV Plug (J1772),2,2,6.2 kW,208 V,30 A,Flo,$2.00 / hr
1339,104887,EV Plug (J1772),2,1,6.7 kW,208 V,32 A,SWTCH,Free: 0-120 mins; $3.00/hr: after 120 mins: Fr...
1340,104887,EV Plug (J1772),2,1,6.7 kW,208 V,32 A,SWTCH,Free: 0-120 mins; $3.00/hr: after 120 mins: Fr...
1400,105099,EV Plug (J1772),2,3,6.2 kW,208 V,30 A,Flo,Cost: Free
1401,105099,EV Plug (J1772),2,1,6.2 kW,208 V,30 A,Flo,Cost: Free
1542,107029,EV Plug (J1772),2,2,6.6 kW,240 V,27 A,ChargePoint,Parking \n 1.5 CAD per hour for first 120 mi...
1668,108549,EV Plug (J1772),2,8,6.6 kW,240 V,27 A,ChargePoint,Parking \n 0 CAD per hour for first 120 mins...
1771,108942,EV Plug (J1772),2,1,6.7 kW,208 V,32 A,SWTCH,Free: 0-120 mins; $3.00/hr: after 120 mins: Fr...
1818,109023,EV Plug (J1772),2,2,6.6 kW,240 V,27 A,ChargePoint,Cost: Free


### Convert Technical Fields to Numeric Values

The original `Kw`, `Volt`, and `Amp` fields contain units as text.

Numeric versions are created while preserving the original source columns for validation.


In [38]:
# Create numeric versions of the technical charging fields

# Remove " kW" from the power values and convert to numbers
plugs_ottawa["kw_numeric"] = pd.to_numeric(
    plugs_ottawa["Kw"].str.replace(" kW", "", regex=False),
    errors="coerce"
)

# Remove " V" from voltage values and convert to numbers
plugs_ottawa["volt_numeric"] = pd.to_numeric(
    plugs_ottawa["Volt"].str.replace(" V", "", regex=False),
    errors="coerce"
)

# Remove " A" from amperage values and convert to numbers
plugs_ottawa["amp_numeric"] = pd.to_numeric(
    plugs_ottawa["Amp"].str.replace(" A", "", regex=False),
    errors="coerce"
)

# Check the cleaned numeric fields
plugs_ottawa[
    [
        "Name",
        "Level",
        "Kw",
        "kw_numeric",
        "Volt",
        "volt_numeric",
        "Amp",
        "amp_numeric"
    ]
].head(20)

,Name,Level,Kw,kw_numeric,Volt,volt_numeric,Amp,amp_numeric
135,EV Plug (J1772),2,12 kW,12.0,240 V,240.0,50 A,50.0
1178,EV Plug (J1772),2,6.2 kW,6.2,208 V,208.0,30 A,30.0
1339,EV Plug (J1772),2,6.7 kW,6.7,208 V,208.0,32 A,32.0
1340,EV Plug (J1772),2,6.7 kW,6.7,208 V,208.0,32 A,32.0
1400,EV Plug (J1772),2,6.2 kW,6.2,208 V,208.0,30 A,30.0
1401,EV Plug (J1772),2,6.2 kW,6.2,208 V,208.0,30 A,30.0
1542,EV Plug (J1772),2,6.6 kW,6.6,240 V,240.0,27 A,27.0
1668,EV Plug (J1772),2,6.6 kW,6.6,240 V,240.0,27 A,27.0
1771,EV Plug (J1772),2,6.7 kW,6.7,208 V,208.0,32 A,32.0
1818,EV Plug (J1772),2,6.6 kW,6.6,240 V,240.0,27 A,27.0


In [40]:
# Review power by charging level
plugs_ottawa.groupby("Level")["kw_numeric"].describe().T

Level,1,2,3
count,0.0,163.00000,28.000000
mean,NaN,7.77681,97.928571
std,NaN,2.81276,85.486377
min,NaN,5.00000,50.000000
25%,NaN,6.60000,50.000000
50%,NaN,6.60000,50.000000
75%,NaN,7.20000,127.500000
max,NaN,19.20000,325.000000


## 11. Station-Level Charging Power

For each Ottawa station, identify the highest reported charging level and maximum power.

This allows stations to be grouped into a simple infrastructure category:

- Level 1
- Level 2
- DC Fast


In [41]:
# Find the highest-power plug record for each station
station_power = (
    plugs_ottawa.groupby("StationId")
    .agg(
        max_kw=("kw_numeric", "max"),
        max_level=("Level", "max")
    )
    .reset_index()
)

# Merge station name and address for context
station_power = station_power.merge(
    stations_ottawa[
        ["Id", "LocName", "Address"]
    ],
    left_on="StationId",
    right_on="Id",
    how="left"
)

# Sort from highest to lowest charging power
station_power = station_power.sort_values(
    "max_kw",
    ascending=False
)

station_power[
    [
        "LocName",
        "Address",
        "max_level",
        "max_kw"
    ]
].head(20)

,LocName,Address,max_level,max_kw
217,"Ottawa - Boundary Rd, ON Supercharger","5329 Boundary Road, Ottawa, ON, K4B 1P6",3,325.0
196,"Ottawa - Campeau Dr, ON Supercharger","8605 Campeau Drive, Ottawa, ON, K2T 0K5",3,325.0
216,"Ottawa - St. Laurent Blvd, ON Supercharger","1055 Saint Laurent Boulevard, Ottawa, ON, K1K 3B1",3,250.0
213,Petro-Canada,"8605 Campeau Drive, Ottawa, ON, K2T 0S4",3,200.0
48,Canadian Tire Ottawa East,"330 Coventry Road, Ottawa, ON, K1K 4S3",3,150.0
191,Donnelly Ford Lincoln,"2496 Bank Street, Ottawa, ON, K1V 0W8",3,120.0
26,"Ottawa, ON Supercharger","50 Rideau Street, Ottawa, ON, K1N 9J7",3,72.0
25,Cadillac Fairview - Rideau Center,"50 Rideau Street, Ottawa, ON, K1N 9J7",3,50.0
170,Stony Swamp Park,"P11 NCC 26 Greenbelt P11, Ottawa, ON, K2R 1H4",3,50.0
163,Myers Barrhaven Subaru,"4151 Strandherd Drive, Ottawa, ON, K2J 6H8",3,50.0


In [42]:
# Create a simple charger category for each station
def charger_category(row):
    if row["max_level"] == 3:
        return "DC Fast"
    elif row["max_level"] == 2:
        return "Level 2"
    elif row["max_level"] == 1:
        return "Level 1"
    else:
        return "Unknown"

# Apply the function
station_power["charger_category"] = station_power.apply(
    charger_category,
    axis=1
)

# Count stations by charger category
station_power["charger_category"].value_counts()

charger_category
Level 2    189
DC Fast     28
Level 1      1
Name: count, dtype: int64

### Capacity by Charger Category

Station counts alone can hide infrastructure capacity, so total ports and average ports per station are also summarized by charger category.

The Ottawa network is strongly Level 2-oriented, while DC fast charging represents a much smaller portion of stations and ports.


In [43]:
# Add total port count to the station-level charger category table
station_power = station_power.merge(
    stations_ottawa[
        ["Id", "TotalPorts"]
    ],
    on="Id",
    how="left"
)

# Summarize station count and total ports by charger category
category_summary = (
    station_power.groupby("charger_category")
    .agg(
        station_count=("Id", "count"),
        total_ports=("TotalPorts", "sum"),
        avg_ports_per_station=("TotalPorts", "mean")
    )
    .reset_index()
)

# Round the average for easier reading
category_summary["avg_ports_per_station"] = (
    category_summary["avg_ports_per_station"].round(2)
)

category_summary

,charger_category,station_count,total_ports,avg_ports_per_station
0,DC Fast,28,110,3.93
1,Level 1,1,1,1.00
2,Level 2,189,717,3.79


## 12. Charging Networks and Pricing

Review the relationship between charging-network operators, charger levels, and published pricing descriptions.

The raw `PriceString` field is retained because pricing rules can include hourly rates, per-minute rates, per-kWh rates, parking fees, free charging, membership pricing, and tiered structures.


In [44]:
# Compare charging networks and their pricing information
network_pricing = (
    plugs_ottawa[
        ["NetworkName", "PriceString"]
    ]
    .drop_duplicates()
    .sort_values("NetworkName")
)

network_pricing

,NetworkName,PriceString
5952,AmbuEnergy,$0.05 / min
9356,AmbuEnergy,$0.04 / min
8860,AmbuEnergy,Cost: Free
11834,ChargeLab,$3.00 / hr
11870,ChargeLab,Cost: Free
...,...,...
9995,NaN,Parking fee only
15752,NaN,Cost: Free
19789,NaN,$10.00 / session
22982,NaN,$2 for 2 hours; $3 for 4 hours


In [45]:
# Count plug records by network
plugs_ottawa["NetworkName"].value_counts(dropna=False)

NetworkName
Flo                    81
ChargePoint            49
NaN                    49
SWTCH                  14
Unknown                 8
Tesla                   8
Circuit Électrique      7
ChargeLab               6
EVStart                 6
AmbuEnergy              6
Hypercharge             4
Ivy                     4
Electrify Canada        3
Petro-Canada            2
Sun Country Highway     2
myEVroute               1
Ford Charge             1
EVConnect               1
Name: count, dtype: int64

In [46]:
# Compare charging level by network
pd.crosstab(
    plugs_ottawa["NetworkName"],
    plugs_ottawa["Level"]
)

Level,2,3
NetworkName,,
AmbuEnergy,6,0
ChargeLab,6,0
ChargePoint,49,0
Circuit Électrique,4,3
EVConnect,1,0
EVStart,6,0
Electrify Canada,0,3
Flo,68,13
Ford Charge,0,1


In [54]:
# Display the original pricing column
# Compare original pricing text with our simplified category
plugs_ottawa[
    [
        "PriceString",
        "price_category"
    ]
].head(20)


,PriceString,price_category
135,Cost: Free,Free
1178,$2.00 / hr,Other / Complex
1339,Free: 0-120 mins; $3.00/hr: after 120 mins: Fr...,Complex / Tiered
1340,Free: 0-120 mins; $3.00/hr: after 120 mins: Fr...,Complex / Tiered
1400,Cost: Free,Free
1401,Cost: Free,Free
1542,Parking \n 1.5 CAD per hour for first 120 mi...,Complex / Tiered
1668,Parking \n 0 CAD per hour for first 120 mins...,Complex / Tiered
1771,Free: 0-120 mins; $3.00/hr: after 120 mins: Fr...,Complex / Tiered
1818,Cost: Free,Free


### Create a Simplified Pricing Category

Because `PriceString` contains many different formats, a derived `price_category` field is created for analysis.

The original pricing text is preserved. The category is only a simplified analytical label and does not replace the source tariff description.


In [60]:
# Create a more accurate pricing category
def price_category(price):

    # Handle missing values
    if pd.isna(price):
        return "Unknown"

    # Convert to lowercase for easier matching
    price_lower = price.lower()

    # Unknown pricing
    if "cost: unknown" in price_lower:
        return "Unknown"

    # Pricing is not listed directly in the dataset
    elif "pricing varies" in price_lower:
        return "Variable / External"

    # Parking fee only
    elif "parking fee only" in price_lower:
        return "Parking Only"

    # Charging is free but parking has a cost
    elif (
        "free charging, pay for parking" in price_lower
        or "free charging, paid parking" in price_lower
    ):
        return "Free Charging / Paid Parking"

    # Complex or tiered pricing rules
    elif (
        "palier" in price_lower
        or "tier" in price_lower
        or "after" in price_lower
        or "first" in price_lower
        or "min fee" in price_lower
        or "max fee" in price_lower
        or "monthly fee" in price_lower
        or "grace period" in price_lower
        or "loitering" in price_lower
        or "until unplugged" in price_lower
        or "$2 for 2 hours" in price_lower
    ):
        return "Complex / Tiered"

    # Free charging
    elif "cost: free" in price_lower or price_lower.strip() == "free":
        return "Free"

    # Pricing based on energy used
    elif "/ kwh" in price_lower or "/kwh" in price_lower:
        return "Per kWh"

    # Pricing by minute
    elif "/ min" in price_lower or "/min" in price_lower:
        return "Per Minute"

    # Pricing by hour
    elif (
        "/ hr" in price_lower
        or "/hr" in price_lower
        or "per hour" in price_lower
    ):
        return "Hourly"

    # Flat fee per session
    elif "/ session" in price_lower or "/session" in price_lower:
        return "Per Session"

    # Anything not captured above
    else:
        return "Other / Complex"

In [61]:
# Re-apply pricing categories
plugs_ottawa["price_category"] = (
    plugs_ottawa["PriceString"]
    .apply(price_category)
)

# Check updated counts
plugs_ottawa["price_category"].value_counts()

price_category
Hourly                          119
Unknown                          46
Complex / Tiered                 33
Free                             32
Per kWh                           7
Per Minute                        5
Variable / External               4
Per Session                       3
Free Charging / Paid Parking      2
Parking Only                      1
Name: count, dtype: int64

### Pricing Classification Validation

The final classification leaves no records in the temporary `Other / Complex` bucket.

Final categories include:

- Hourly
- Unknown
- Complex / Tiered
- Free
- Per kWh
- Per Minute
- Variable / External
- Per Session
- Free Charging / Paid Parking
- Parking Only


In [62]:
# Check whether any pricing descriptions remain unclassified
plugs_ottawa[
    plugs_ottawa["price_category"] == "Other / Complex"
][
    ["PriceString", "price_category"]
]

,PriceString,price_category


# Key Findings and Next Steps

## Key Findings

- The Ottawa infrastructure subset contains **218 charging locations**.
- All 218 stations are represented in both the port and plug datasets.
- Station capacity is highly consistent across the three source tables: **217 of 218 stations** have matching station, port, and plug counts.
- Ottawa's charging infrastructure is dominated by **Level 2** stations, with a smaller DC fast-charging segment.
- Plug-level technical fields provide usable charger power, voltage, amperage, connector, and network information.
- Pricing descriptions can be organized into useful categories while preserving the original tariff text.
- Location names can provide valuable real-world context, such as retail, arenas, malls, dealerships, Park & Ride facilities, and other destination types.

## Intended Use

The purpose of this notebook is to prepare a trusted **station-level enrichment dataset** that can later be linked to the cleaned Ottawa EV charging-session dataset.

Potential enrichment fields include:

- actual port capacity
- charger category
- maximum charging power
- plug type
- charging network
- pricing category
- location name and destination context
- latitude and longitude

These fields may help explain differences in session duration, energy delivered, site utilization, charging behaviour, and tariff-informed estimated charging revenue.

## Next Step

Return to the main Ottawa EV charging analysis and decide which validated station-level fields should be merged into the cleaned usage dataset. A separate geospatial notebook can then investigate nearby amenities and location context in more detail.
